# Step 2: Pure ML Training & Evaluation Pipeline

Run this notebook after completing Step 1 and generating `data/harmonized_asd_3450.csv`. The original training and evaluation code is preserved.

In [ ]:
%pip install -q xgboost statsmodels scikit-learn scipy pandas numpy joblib

In [ ]:
# If you are starting a fresh Colab session, upload the output from Step 1.
# Choose data/harmonized_asd_3450.csv when prompted.
from google.colab import files
import os

if not os.path.exists('data/harmonized_asd_3450.csv'):
    uploaded = files.upload()
    os.makedirs('data', exist_ok=True)
    for name in uploaded:
        if name.endswith('harmonized_asd_3450.csv'):
            os.replace(name, 'data/harmonized_asd_3450.csv')
            print('Dataset saved to data/harmonized_asd_3450.csv')
            break
    else:
        raise FileNotFoundError('Upload harmonized_asd_3450.csv from Step 1.')


In [ ]:
"""
Step 2: Pure ML Training & Evaluation Pipeline
==============================================
Run:
    python 2_train_model.py
Outputs in ./results:
    xgb_platt.joblib
    table1_holdout.csv
    table2_calibration.csv
    significance.csv
    summary.json
"""

import json
import os
import warnings

import joblib
import numpy as np
import pandas as pd
from scipy.stats import binomtest, wilcoxon
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, brier_score_loss, confusion_matrix,
                             f1_score, log_loss, matthews_corrcoef,
                             precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import (ParameterSampler, RepeatedStratifiedKFold,
                                     StratifiedKFold, cross_val_predict,
                                     train_test_split)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from statsmodels.stats.multitest import multipletests
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

SEED = 42
ITEMS = [f"A{i}" for i in range(1, 11)]
CAT_COLS = ["sex", "ethnicity", "jaundice", "family_history", "respondent", "age_group"]
NUM_COLS = ["age", "item_sum"]
DEMO_CAT = ["sex", "ethnicity", "jaundice", "family_history", "respondent", "age_group"]


def make_preprocessor(use_items=True, scale_items=False):
    nums = ["age", "item_sum"] if use_items else ["age"]
    item_trans = StandardScaler() if scale_items else "passthrough"
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), nums),
        ("items", item_trans, ITEMS if use_items else []),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_COLS),
    ])


def xgb_base():
    return XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.9,
        colsample_bytree=0.9, objective="binary:logistic", eval_metric="logloss",
        random_state=SEED, n_jobs=1, tree_method="hist"
    )


def make_pipe(model, scale_items=False):
    return Pipeline([("prep", make_preprocessor(use_items=True, scale_items=scale_items)), ("clf", model)])


def baselines():
    return {
        "Decision tree": DecisionTreeClassifier(max_depth=5, min_samples_leaf=4, random_state=SEED),
        "Random forest": RandomForestClassifier(n_estimators=200, max_depth=6, min_samples_leaf=4,
                                                random_state=SEED, n_jobs=1),
        "SVM": SVC(kernel="rbf", C=1.5, probability=True, random_state=SEED),
        "MLP": MLPClassifier(hidden_layer_sizes=(64, 32), early_stopping=True, random_state=SEED, max_iter=500),
        "KNN": KNeighborsClassifier(n_neighbors=19),
        "Logistic regression": LogisticRegression(C=0.2, max_iter=2000, random_state=SEED),
    }


def feature_frame(df):
    return df[ITEMS + NUM_COLS + CAT_COLS].copy()


def tune_xgb(X, y, n_iter=10):
    space = {
        "max_depth": [3, 4, 5, 6],
        "learning_rate": [0.03, 0.05, 0.1],
        "n_estimators": [150, 300, 500],
        "min_child_weight": [1, 3, 5],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.7, 1.0],
        "reg_lambda": [1, 5],
    }
    cands = [{}] + list(ParameterSampler(space, n_iter=n_iter, random_state=SEED))
    cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
    best, best_s = {}, -np.inf
    print(f"Running random search over {len(cands)} configurations...")
    for i, params in enumerate(cands):
        m = make_pipe(clone(xgb_base()).set_params(**params))
        p = cross_val_predict(m, X, y, cv=cv, method="predict_proba")[:, 1]
        s = f1_score(y, p > 0.5) - 0.01 * log_loss(y, np.clip(p, 1e-6, 1 - 1e-6))
        print(f"  config {i:02d} score={s:.4f} {params or 'defaults'}")
        if s > best_s:
            best, best_s = params, s
    print(f"-> Selected best parameters: {best or 'defaults'} (score: {best_s:.4f})")
    return best


def _logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def fit_platt(oof_p, y):
    lr = LogisticRegression(C=1e6, max_iter=1000).fit(_logit(oof_p).reshape(-1, 1), y)
    fn = lambda p: np.clip(lr.predict_proba(_logit(p).reshape(-1, 1))[:, 1], 0.002, 0.998)
    fn.coef, fn.intercept = float(lr.coef_[0][0]), float(lr.intercept_[0])
    return fn


def fit_isotonic(oof_p, y):
    iso = IsotonicRegression(out_of_bounds="clip").fit(oof_p, y)
    return lambda p: np.clip(iso.predict(p), 0.002, 0.998)


def ece_score(y, p, bins=10):
    y = np.asarray(y)
    idx = np.clip(np.digitize(p, np.linspace(0, 1, bins + 1)[1:-1]), 0, bins - 1)
    return sum((idx == b).mean() * abs(y[idx == b].mean() - p[idx == b].mean())
               for b in range(bins) if (idx == b).any())


def metrics(y, p, thr=0.5):
    y = np.asarray(y)
    yhat = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yhat).ravel()
    return {
        "accuracy": accuracy_score(y, yhat),
        "precision": precision_score(y, yhat),
        "recall": recall_score(y, yhat),
        "specificity": tn / (tn + fp),
        "f1": f1_score(y, yhat),
        "roc_auc": roc_auc_score(y, p),
        "mcc": matthews_corrcoef(y, yhat),
        "log_loss": log_loss(y, np.clip(p, 1e-6, 1 - 1e-6)),
        "brier": brier_score_loss(y, p),
        "ece": ece_score(y, p),
        "errors": int(fp + fn),
    }


def mcnemar_exact(y, pa, pb):
    a_ok, b_ok = pa == y, pb == y
    n01, n10 = int(((~a_ok) & b_ok).sum()), int((a_ok & (~b_ok)).sum())
    n = n01 + n10
    return (1.0 if n == 0 else binomtest(min(n01, n10), n, 0.5).pvalue), n01, n10


def bootstrap_f1_diff(y, pa, pb, n_boot=2000):
    rng, y = np.random.default_rng(SEED), np.asarray(y)
    d = []
    for _ in range(n_boot):
        i = rng.integers(0, len(y), len(y))
        d.append(f1_score(y[i], pa[i]) - f1_score(y[i], pb[i]))
    return np.percentile(d, [2.5, 97.5])


def repeated_cv_f1(models, X, y, n_splits=5, n_repeats=2):
    scores = {k: [] for k in models}
    for tr, va in RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=SEED).split(X, y):
        Xtr, ytr = X.iloc[tr], y.iloc[tr]
        for name, m in models.items():
            scores[name].append(f1_score(y.iloc[va], clone(m).fit(Xtr, ytr).predict(X.iloc[va])))
    return scores


def threshold_rule(df):
    print("\n=== Descriptive Threshold Rule ===")
    rows, correct = [], 0.0
    for g, d in df.groupby("age_group"):
        t, acc = max(((t, ((d["item_sum"] >= t).astype(int) == d["label"]).mean()) for t in range(11)),
                     key=lambda x: x[1])
        rows.append({"age_group": g, "n": len(d), "best_threshold": int(t), "accuracy": round(float(acc), 4)})
        correct += acc * len(d)
    print(pd.DataFrame(rows).to_string(index=False))
    overall = float(correct / len(df))
    print(f"Per-age-group threshold accuracy: {overall:.4f}")
    return rows, overall


if __name__ == "__main__":
    data_path = os.path.join("data", "harmonized_asd_3450.csv")
    if not os.path.exists(data_path):
        raise FileNotFoundError(f"Missing {data_path}. Please run 'python 1_download_data.py' first!")

    out_dir = "results"
    os.makedirs(out_dir, exist_ok=True)

    print(f"[1/6] Loading harmonized dataset from {data_path}...")
    df = pd.read_csv(data_path)
    print(f"  Loaded {len(df)} unique records (A={int((df['source']=='A').sum())}, B={int((df['source']=='B').sum())})")

    rule_rows, rule_acc = threshold_rule(df)

    print("\n[2/6] Creating Stratified 80/20 Train/Test Split...")
    strat = df["source"] + "_" + df["label"].astype(str)
    tr_i, te_i = train_test_split(df.index, test_size=0.2, random_state=SEED, stratify=strat)
    train, test = df.loc[tr_i], df.loc[te_i]
    Xtr, ytr, Xte, yte = feature_frame(train), train["label"], feature_frame(test), test["label"]
    print(f"  Split sizes: train={len(train)} (ASD={int(ytr.sum())}) | test={len(test)}")

    print("\n[3/6] Tuning XGBoost & Fitting Models...")
    best = tune_xgb(Xtr, ytr, n_iter=10)
    xgb = make_pipe(clone(xgb_base()).set_params(**best))
    models = {"XGBoost": xgb}
    for k, clf in baselines().items():
        models[k] = make_pipe(clf, scale_items=(k in ("KNN", "Logistic regression", "SVM", "MLP")))

    probs, results = {}, {}
    for name, m in models.items():
        m.fit(Xtr, ytr)
        probs[name] = m.predict_proba(Xte)[:, 1]
        results[name] = metrics(yte, probs[name])
        print(f"  {name:<22} F1={results[name]['f1']:.4f}  errors={results[name]['errors']}")

    t1 = pd.DataFrame(results).T.sort_values("f1", ascending=False)
    t1.to_csv(os.path.join(out_dir, "table1_holdout.csv"))
    print("\n=== TABLE I: Hold-out Evaluation ===")
    print(t1[["accuracy", "precision", "recall", "f1", "roc_auc", "mcc", "errors"]].round(4).to_string())

    print("\n[4/6] Statistical Significance & Cross-Validation...")
    y_arr = yte.values
    xgb_pred = (probs["XGBoost"] >= 0.5).astype(int)
    cv_f1 = repeated_cv_f1(models, Xtr, ytr)
    names = [k for k in models if k != "XGBoost"]
    mc_p, wl_p, cis = [], [], []
    for n in names:
        pred = (probs[n] >= 0.5).astype(int)
        mc_p.append(mcnemar_exact(y_arr, xgb_pred, pred)[0])
        try:
            wl_p.append(wilcoxon(cv_f1["XGBoost"], cv_f1[n]).pvalue)
        except ValueError:
            wl_p.append(1.0)
        cis.append(bootstrap_f1_diff(y_arr, xgb_pred, pred).round(4).tolist())

    sig = pd.DataFrame({
        "baseline": names,
        "mcnemar_p": mc_p,
        "mcnemar_holm": multipletests(mc_p, method="holm")[1],
        "wilcoxon_p": wl_p,
        "wilcoxon_holm": multipletests(wl_p, method="holm")[1],
        "cv_median_f1_xgb": np.median(cv_f1["XGBoost"]),
        "cv_median_f1_base": [np.median(cv_f1[n]) for n in names],
        "f1_diff_ci95": cis,
    })
    sig.to_csv(os.path.join(out_dir, "significance.csv"), index=False)
    print(sig.round(4).to_string(index=False))

    print("\n[5/6] Probability Calibration...")
    oof = cross_val_predict(clone(xgb), Xtr, ytr, cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
                            method="predict_proba")[:, 1]
    platt = fit_platt(oof, ytr.values)
    iso = fit_isotonic(oof, ytr.values)
    p_raw = probs["XGBoost"]
    t2 = pd.DataFrame({
        "uncalibrated": metrics(y_arr, p_raw),
        "platt": metrics(y_arr, platt(p_raw)),
        "isotonic": metrics(y_arr, iso(p_raw)),
    }).T[["log_loss", "ece", "brier", "f1"]]
    t2.to_csv(os.path.join(out_dir, "table2_calibration.csv"))
    print(t2.round(4).to_string())

    print("\n[6/6] Saving Artifacts for Local Streamlit Dashboard...")
    demo_cols = ["age"] + DEMO_CAT
    demo_pipe = Pipeline([
        ("prep", ColumnTransformer([
            ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), ["age"]),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), DEMO_CAT),
        ])),
        ("clf", XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.04, random_state=SEED, n_jobs=1)),
    ]).fit(Xtr[demo_cols], ytr)
    demo_auc = float(roc_auc_score(yte, demo_pipe.predict_proba(Xte[demo_cols])[:, 1]))

    joblib.dump({
        "model": xgb,
        "platt_coef": platt.coef,
        "platt_intercept": platt.intercept,
        "features": list(Xtr.columns),
    }, os.path.join(out_dir, "xgb_platt.joblib"))

    with open(os.path.join(out_dir, "summary.json"), "w") as f:
        json.dump({
            "best_xgb_params": best,
            "rule_accuracy": rule_acc,
            "rule_rows": rule_rows,
            "n_train": len(train),
            "n_test": len(test),
            "demographics_only_auc": demo_auc,
        }, f, indent=2)

    print(f"\nSuccess! Pure ML artifacts saved to '{out_dir}/'. Run 'streamlit run 3_app.py' next.")